In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch

# Test if Model works in general

In [3]:
from genpp.models.chen import FcChenModel

In [5]:
model = FcChenModel(
    in_features=29,
    out_features=2,
    meta_features=5,
    height=5,
    width=5,
    noise_dim=5,
    hidden_dim_std=50,
    hidden_dim_decoder=100,
    embedding_dim=5,
    n_samples_train=50,
)

In [ ]:
test_input = torch.cat(
    (torch.randn(8, 5, 5, 2 * 29 + 5), torch.randint(0, 5, (8, 5, 5, 1))), dim=-1
)
test_out = model.forward(test_input)
test_out.shape  # Shape is good 🥳

torch.Size([8, 50, 5, 5, 2])

In [8]:
model

FcChenModel(
  (final_activation): Identity()
  (embedding): Embedding(25, 5)
  (_mean_model): Sequential(
    (0): LocallyConnected2D()
    (1): Rearrange('b h w o -> b 1 h w o')
  )
  (_std_model): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=725, out_features=50, bias=True)
    (2): ELU(alpha=1.0)
    (3): Linear(in_features=50, out_features=5, bias=True)
    (4): Softplus(beta=1.0, threshold=20.0)
    (5): Rearrange('b noise_dim -> b 1 noise_dim')
  )
  (_noise_decoder): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1705, out_features=100, bias=True)
    (2): ELU(alpha=1.0)
    (3): Linear(in_features=100, out_features=50, bias=True)
    (4): Rearrange('(b n) (h w o) -> b n h w o', n=50, h=5, w=5, o=2)
  )
)

In [9]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [10]:
num_params = count_parameters(model)
print(f"Number of trainable parameters: {num_params:,}")

Number of trainable parameters: 213,830


In [11]:
from genpp.models.loss import EnergyScore

In [12]:
es = EnergyScore()
test_y = torch.randn(8, 5, 5, 2)  # Example target tensor
es(test_out, test_y)

tensor([[28.2654, 20.9128],
        [30.2670, 23.2879],
        [29.1640, 31.2433],
        [21.2035, 20.4556],
        [22.5275, 30.4445],
        [29.7365, 26.4877],
        [26.1146, 28.5335],
        [27.8131, 26.1854]], grad_fn=<SubBackward0>)

## Test CNNChen

In [13]:
import torch

from genpp.models.chen import CNNChenModel

In [ ]:
test_data = torch.cat(
    (torch.randn(8, 32, 48, 2 * 29 + 5), torch.randint(0, 32 * 48, (8, 32, 48, 1))), dim=-1
)  # Example input tensor
model = CNNChenModel(
    in_features=29,
    out_features=2,
    meta_features=5,
    noise_dim=1,
    embedding_dim=5,
    n_samples_train=50,
    width=48,
    height=32,
    padding=(2, 2, 3, 3),
)

In [ ]:
res = model(test_data)
res.shape  # Should be [8, 5, 28, 42, 2], due to the padding being removed in the output

torch.Size([8, 50, 28, 42, 2])